# CEG-WM Content V9 — repeatable comparison run

This Colab notebook performs one fresh Content V9 formal-run invocation per top-to-bottom execution. It clones the complete Content-V9-Evidence execution snapshot, records its actual commit/branch/clean identity, installs that checkout, reads the two Colab Secrets, and invokes the checkout's formal runner exactly once.

Every execution uses a new create-only Drive sink named `Content-V9-<shortcommit>-<utc>` under `/content/drive/MyDrive/CEG-WM/Content/`. The runner owns the checkpoint and terminal artifacts. The final cell only inspects existing output and never launches the runner.

Before running, add Colab Secrets `CEG_WM_ROOT_KEY` and `HF_TOKEN`, select a GPU runtime, and execute all cells once in order. To perform another independent comparison run, start again from the first cell; a name collision stops safely without overwrite, retry, resume, or fallback.

## 1. Mount Google Drive

Drive mounting is the notebook's first external action.

In [ ]:
import json
from google.colab import drive

FAILURE_PREFIX = "CEGWM_CONTENT_V9_HANDOFF_FAILURE"
HANDOFF_FAILED = False
RUNNER_ATTEMPTED = False
DRIVE_TARGET = None
RUNNER_RC = None
EXECUTION_COMMIT = None
CHECKOUT_BRANCH = None
_ALLOWED_ERROR_CLASSES = {
    "CalledProcessError", "FileExistsError", "ImportError",
    "ModuleNotFoundError", "OSError", "RuntimeError",
    "TypeError", "UnicodeDecodeError", "ValueError",
}

def handoff_fail(stage, error):
    global HANDOFF_FAILED
    if HANDOFF_FAILED:
        return
    HANDOFF_FAILED = True
    error_class = type(error).__name__
    if error_class not in _ALLOWED_ERROR_CLASSES:
        error_class = "OtherOperationalError"
    payload = {
        "status": "operational_failure",
        "stage": stage,
        "error_class": error_class,
    }
    print(FAILURE_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":")), flush=True)

try:
    drive.mount("/content/drive")
except BaseException as error:
    handoff_fail("drive_mount", error)

## 2. Clone the specified branch and record checkout identity

The checkout record contains only the requested branch, actual commit, short commit, and clean-state identity. A dirty or unexpected checkout stops before installation.

In [ ]:
import pathlib
import subprocess
import sys
from datetime import datetime, timezone

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
BRANCH = "Content-V9-Evidence"
RUNNER_MODULE = "experiments.run_content_v9_stability"
def git(*args):
    return subprocess.run(
        ["git", *args], cwd=REPO, check=True, capture_output=True, text=True,
    ).stdout.strip()

if not HANDOFF_FAILED:
    try:
        SESSION_UTC = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        REPO = pathlib.Path(f"/content/cegwm-content-v9-source-{SESSION_UTC}")
        if REPO.exists():
            raise FileExistsError(f"fresh source path required: {REPO}")
        subprocess.run(
            ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO)],
            check=True,
        )
        EXECUTION_COMMIT = git("rev-parse", "HEAD")
        SHORT_COMMIT = EXECUTION_COMMIT[:7]
        CHECKOUT_BRANCH = git("branch", "--show-current")
        CHECKOUT_CLEAN = git("status", "--porcelain") == ""
        CHECKOUT_RECORD = {
            "requested_branch": BRANCH,
            "checkout_branch": CHECKOUT_BRANCH,
            "commit": EXECUTION_COMMIT,
            "short_commit": SHORT_COMMIT,
            "clean": CHECKOUT_CLEAN,
        }
        print("CEGWM_CONTENT_V9_CHECKOUT " + json.dumps(CHECKOUT_RECORD, sort_keys=True))
        if CHECKOUT_BRANCH != BRANCH or not CHECKOUT_CLEAN:
            raise RuntimeError("unexpected branch identity or dirty checkout")
    except BaseException as error:
        handoff_fail("source_checkout", error)

## 3. Install the checkout's declared project dependencies

Installation uses the cloned checkout itself. The checkout identity is recorded again afterward so the runner receives the exact installed source revision.

In [ ]:
if not HANDOFF_FAILED:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", str(REPO)],
            check=True,
        )
        POST_INSTALL_COMMIT = git("rev-parse", "HEAD")
        POST_INSTALL_BRANCH = git("branch", "--show-current")
        POST_INSTALL_CLEAN = git("status", "--porcelain") == ""
        POST_INSTALL_RECORD = {
            "checkout_branch": POST_INSTALL_BRANCH,
            "commit": POST_INSTALL_COMMIT,
            "clean": POST_INSTALL_CLEAN,
        }
        print("CEGWM_CONTENT_V9_POST_INSTALL " + json.dumps(POST_INSTALL_RECORD, sort_keys=True))
        if (
            POST_INSTALL_COMMIT != EXECUTION_COMMIT
            or POST_INSTALL_BRANCH != BRANCH
            or not POST_INSTALL_CLEAN
        ):
            raise RuntimeError("checkout identity changed during installation")
    except BaseException as error:
        handoff_fail("dependency_install", error)

## 4–7. Read Secrets, create a unique Drive target, and call the formal runner once

The target format is `/content/drive/MyDrive/CEG-WM/Content/Content-V9-<shortcommit>-<utc>/`. Secrets are passed only to the child runner environment and removed from notebook variables immediately afterward. Runner stdout/stderr are inherited directly: the notebook does not accumulate, truncate, reclassify, or parse them. The runner is the sole writer of run state, checkpoint pairs, and the terminal ZIP/SHA pair under this Drive sink.

In [ ]:
import os
from google.colab import userdata

root_key = ""
hf_token = ""
runner_env = None
if not HANDOFF_FAILED:
    try:
        if RUNNER_ATTEMPTED:
            raise RuntimeError("runner was already attempted in this runtime")
        RUNNER_ATTEMPTED = True
        root_key = userdata.get("CEG_WM_ROOT_KEY")
        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(root_key, str) or not root_key.strip():
            raise RuntimeError("CEG_WM_ROOT_KEY Colab Secret is required")
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError("HF_TOKEN Colab Secret is required")
        RUN_UTC = SESSION_UTC
        RUN_DIRECTORY_NAME = f"Content-V9-{SHORT_COMMIT}-{RUN_UTC}"
        DRIVE_CONTENT_ROOT = pathlib.Path("/content/drive/MyDrive/CEG-WM/Content")
        DRIVE_TARGET = DRIVE_CONTENT_ROOT / RUN_DIRECTORY_NAME
        LOCAL_WORK_ROOT = pathlib.Path("/content") / (RUN_DIRECTORY_NAME + "-local")
        if DRIVE_TARGET.exists() or LOCAL_WORK_ROOT.exists():
            raise FileExistsError("create-only run destination already exists")
        secret_markers = ("TOKEN", "KEY", "SECRET", "PASSWORD", "CREDENTIAL")
        runner_env = {
            name: value for name, value in os.environ.items()
            if not any(marker in name.upper() for marker in secret_markers)
        }
        runner_env["CEG_WM_ROOT_KEY"] = root_key
        runner_env["HF_TOKEN"] = hf_token
        root_key = ""
        hf_token = ""
        RUNNER_COMMAND = [
            sys.executable, "-m", RUNNER_MODULE,
            "--repo-root", str(REPO),
            "--expected-exact", EXECUTION_COMMIT,
            "--local-work-root", str(LOCAL_WORK_ROOT),
            "--artifact-sink", str(DRIVE_TARGET),
        ]
        print("CEGWM_CONTENT_V9_RUN_TARGET " + json.dumps({
            "branch": CHECKOUT_BRANCH,
            "commit": EXECUTION_COMMIT,
            "drive_target": str(DRIVE_TARGET),
            "run_utc": RUN_UTC,
        }, sort_keys=True))
        RUNNER_RC = subprocess.run(
            RUNNER_COMMAND, cwd=REPO, env=runner_env, check=False,
        ).returncode
        print("CEGWM_CONTENT_V9_RUNNER_RETURN " + json.dumps({
            "drive_target": str(DRIVE_TARGET),
            "runner_rc": RUNNER_RC,
        }, sort_keys=True))
    except BaseException as error:
        handoff_fail("formal_runner", error)
    finally:
        root_key = ""
        hf_token = ""
        if runner_env is not None:
            runner_env.pop("CEG_WM_ROOT_KEY", None)
            runner_env.pop("HF_TOKEN", None)
        runner_env = None

## 8. Inspect existing Drive artifacts only

This final cell contains no runner command. It checks the run directory already written by the previous cell, validates complete ZIP/SHA filename bindings, and reports either the terminal pair or the available checkpoint pairs. It does not retry, resume, or fall back.

In [ ]:
import json
import pathlib
import re

REFERENCE_ARCHIVE_SHA256 = "f0ee27da408ca45042f0a4b907b34d38d429d72ca30b944728c82ba9101de39f"
REFERENCE_RECEIPT_SHA256 = "24ae3779294f8123e8da9f603d4f614f45c815840c622a12844f6106351a1570"
REFERENCE_RESULT_SHA256 = "0282db7c40ab4dd9ba663abc3047575c7f86425f8aab4b22ded6b6e28f9c4273"
RUN_ID = "content-v9-stability-9bc8a94c1d02-63c17e8200a9-805bc21e173a"
ARTIFACT_FAILURE_PREFIX = "CEGWM_CONTENT_V9_ARTIFACT_FAILURE"
prior_failure = bool(globals().get("HANDOFF_FAILED", True))
artifact_failed = False

def artifact_fail(stage, error):
    global artifact_failed
    if artifact_failed:
        return
    artifact_failed = True
    error_class = type(error).__name__
    if error_class not in globals().get("_ALLOWED_ERROR_CLASSES", set()):
        error_class = "OtherOperationalError"
    payload = {
        "status": "operational_failure",
        "stage": stage,
        "error_class": error_class,
    }
    print(ARTIFACT_FAILURE_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":")), flush=True)

def inspect_pair(archive_path):
    sidecar_path = pathlib.Path(str(archive_path) + ".sha256")
    if not archive_path.is_file() or not sidecar_path.is_file():
        raise RuntimeError(f"incomplete artifact pair: {archive_path.name}")
    match = re.fullmatch(
        r"([0-9a-f]{64})  ([^\s]+)\n",
        sidecar_path.read_text(encoding="ascii"),
    )
    if match is None or match.group(2) != archive_path.name:
        raise RuntimeError(f"invalid sidecar binding: {sidecar_path.name}")
    return {
        "archive_path": str(archive_path),
        "sidecar_path": str(sidecar_path),
        "declared_sha256": match.group(1),
    }

if prior_failure:
    artifact_fail("artifact_check_skipped_after_prior_failure", RuntimeError())
else:
    try:
        drive_target = globals().get("DRIVE_TARGET")
        if not isinstance(drive_target, pathlib.Path):
            raise RuntimeError("Drive target was not initialized")
        run_directory = drive_target / RUN_ID
        terminal_archive = run_directory / (RUN_ID + ".zip")
        terminal_sidecar = pathlib.Path(str(terminal_archive) + ".sha256")
        if terminal_archive.exists() or terminal_sidecar.exists():
            artifact_kind = "terminal"
            artifact_pairs = [inspect_pair(terminal_archive)]
        else:
            checkpoint_archives = sorted(run_directory.glob(RUN_ID + ".checkpoint-*.zip"))
            checkpoint_sidecars = sorted(run_directory.glob(RUN_ID + ".checkpoint-*.zip.sha256"))
            expected_sidecars = {pathlib.Path(str(path) + ".sha256") for path in checkpoint_archives}
            if not checkpoint_archives or set(checkpoint_sidecars) != expected_sidecars:
                raise RuntimeError("no complete terminal or checkpoint artifact pair exists")
            artifact_kind = "checkpoint"
            artifact_pairs = [inspect_pair(path) for path in checkpoint_archives]
        artifact_receipt = {
            "status": "existing_drive_artifacts_ready",
            "artifact_kind": artifact_kind,
            "branch": globals().get("CHECKOUT_BRANCH"),
            "commit": globals().get("EXECUTION_COMMIT"),
            "drive_target": str(drive_target),
            "runner_rc": globals().get("RUNNER_RC"),
            "pairs": artifact_pairs,
            "reference_archive_sha256": REFERENCE_ARCHIVE_SHA256,
            "reference_receipt_sha256": REFERENCE_RECEIPT_SHA256,
            "reference_result_sha256": REFERENCE_RESULT_SHA256,
        }
        print("CEGWM_CONTENT_V9_EXISTING_ARTIFACTS " + json.dumps(artifact_receipt, sort_keys=True))
    except BaseException as error:
        artifact_fail("artifact_pair_validation", error)

## Stop boundary

Return the final existing-artifact receipt and its referenced Drive paths for independent comparison. This notebook does not rerun failed work automatically, alter denominators or Gates, compare scientific outcomes, adjudicate the result, or overwrite an earlier run.